# Песочница инструментов агента

Меняйте аргументы в ячейках и запускайте (Shift+Enter). Каждая ячейка вызывает инструмент так же, как его будет вызывать агент, и показывает результат таблицей.

- `ok=False` — ошибка: `error_class` и сообщение покажутся вместо таблицы.
- Пустая таблица — «вариантов нет» (распродано, закрыто, не подходит под условия).
- Даты каталога: 2026-11-01 … 2027-10-31.

In [ ]:
import sys
from datetime import date, datetime
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "harness").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display

from synth.tools import RankAreasArgs, SearchHotelsArgs, SearchTransfersArgs, SearchTransportArgs, build_tools

pd.set_option("display.max_colwidth", 80)
T = build_tools()

async def call(name, args, cols=None):
    res = await T[name](args)
    print(f"{name}: ok={res.ok}, найдено {len(res.data) if res.ok else '—'}, {res.latency_ms} мс, call_id={res.call_id}")
    if not res.ok:
        print(f"ОШИБКА {res.error_class}: {res.data.get('message')}")
        if res.data.get("known"):
            print("Известные значения:", res.data["known"])
        return res
    df = pd.json_normalize(res.data)
    display(df[[c for c in cols if c in df.columns]] if cols and len(df) else df)
    return res

## 1. Отели — `search_hotels`

In [ ]:
res = await call("search_hotels", SearchHotelsArgs(
    area="Сириус",                 # id, название или синоним; None — по всем районам
    checkin=date(2026, 11, 10),
    checkout=date(2026, 11, 17),
    adults=2,
    children_ages=[7],
    pets=True,
    max_total=None,                # бюджет на проживание за весь срок, ₽
    meal=None,                     # none | breakfast | half_board | full_board | all_inclusive
    refundable=None,               # True — только с бесплатной отменой
    min_stars=None,
    amenities=[],                  # например ["pool", "kids_club"]
), cols=["hotel.name", "facts.area", "facts.property_type", "facts.stars", "hotel.rating", "room", "meal",
         "facts.cancellation", "price_total", "facts.distance_to_sea_m", "facts.distance_to_lift_m"])

In [ ]:
# подробности первого найденного отеля: разбивка цены, оценки из отзывов, ранний заезд
res.data[0]["facts"] if res.ok and res.data else None

## 2. Транспорт — `search_transport`

In [ ]:
res = await call("search_transport", SearchTransportArgs(
    origin_city="из Москвы",       # «Питер», «СПб», «Екатеринбург», «Краснодар»…
    travel_date=date(2026, 11, 10),
    direction="outbound",          # outbound — в Сочи, return — из Сочи
    kinds=[],                      # пусто — все; или ["flight"], ["train", "bus"]…
    adults=2,
    children_ages=[7],
    direct_only=False,
), cols=["kind", "carrier", "from_city", "to_city", "depart", "arrive", "duration_hours", "stops", "price_total"])

## 3. Трансферы — `search_transfers`

In [ ]:
res = await call("search_transfers", SearchTransfersArgs(
    from_place="аэропорт",         # «вокзал Адлер», «Сочи вокзал», hotel_id, район
    to_place="Роза Хутор",
    when=datetime(2026, 12, 20, 14, 0),
    adults=2,
    children_ages=[7],
    pets=False,
), cols=["kind", "option_id", "from_place", "to_place", "minutes", "price_total", "facts.km", "facts.arrive_at"])

## 4. Районы — `rank_areas`

In [ ]:
res = await call("rank_areas", RankAreasArgs(
    month=2,
    origin_city="Москва",
    adults=2,
    children_ages=[7],
    nights=7,
    wishes=["горы/лыжи"],          # море, горы/лыжи, с детьми, тихо, бюджетно, тусовка, оздоровление,
                                   # экскурсии, активный отдых, романтика, люкс
    max_total=None,                # бюджет поездки целиком
), cols=["name", "score", "air_temp_c", "sea_temp_c", "travel_hours", "est_total_price", "why", "facts.poi"])

## 5. Ошибки и крайние случаи

In [ ]:
_ = await call("search_hotels", SearchHotelsArgs(area="Анапа", checkin=date(2026, 11, 10), checkout=date(2026, 11, 17)))

In [ ]:
_ = await call("search_hotels", {"checkin": "2026-11-17", "checkout": "2026-11-10"})   # даты перепутаны

In [ ]:
# новогодний пик: в Сириусе с животными всё раскуплено — честное «нет вариантов»
_ = await call("search_hotels", SearchHotelsArgs(area="Сириус", checkin=date(2026, 12, 30), checkout=date(2027, 1, 2), pets=True))